# Population Divergence and Admixture Simulation

This notebook simulates an ancestral population that splits into two daughter populations, which later contribute to an admixed population. The simulation uses SLiM5 to model a recombining genome and outputs a tree sequence that is analyzed using tskit.

## Model Description

1. **Ancestral Population**: A population of size `Na` exists from the start
2. **Population Split**: At time `Tsplit`, the ancestral population splits into two daughter populations of sizes `N1` and `N2`
3. **Divergence**: The two populations diverge without gene flow until time `Tadmix`
4. **Admixture**: At time `Tadmix`, a third population of size `NAdmix` is created by merging a fraction `f` from population 1 and `(1-f)` from population 2


## Requirements

- **SLiM**: Version 4.0 or later (tested with SLiM 4.0+)
- **Python**: Version 3.8 or later
- **Python packages**: tskit, numpy, matplotlib, pandas
## Parameters

- `Na`: Ancestral population size
- `N1`: Size of daughter population 1
- `N2`: Size of daughter population 2
- `NAdmix`: Size of admixed population
- `Tsplit`: Generation at which population split occurs
- `Tadmix`: Generation at which admixture occurs
- `f`: Admixture fraction from population 1 (1-f from population 2)
- `genome_length`: Length of simulated genome in base pairs
- `recomb_rate`: Recombination rate per base pair per generation
- `mutation_rate`: Mutation rate per base pair per generation

In [ ]:
# Import required libraries
import subprocess
import os
import tempfile
import tskit
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# Display settings
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10

## Define Simulation Parameters

In [ ]:
# Population sizes
Na = 1000       # Ancestral population size
N1 = 800        # Daughter population 1 size
N2 = 1200       # Daughter population 2 size
NAdmix = 1000   # Admixed population size

# Timing (in generations)
# Note: We use recapitation to simulate the burn-in period for mutation-drift equilibrium
Tburn = 10 * Na  # Burn-in time for ancestral population (10*Na generations)
Tsplit = 1000   # Generation at which split occurs (after burn-in)
Tadmix = 2000   # Generation at which admixture occurs (after burn-in)
Tend = 2500     # Final generation (after burn-in)

# Admixture proportion
f = 0.6         # Fraction from population 1 (0.4 from population 2)

# Genome parameters
genome_length = 1e6      # 1 Mb genome
recomb_rate = 1e-8       # Recombination rate per bp per generation
mutation_rate = 1e-8     # Mutation rate per bp per generation

# Output file
output_trees = "simulation_output.trees"

print("Simulation Parameters:")
print(f"  Ancestral pop size (Na): {Na}")
print(f"  Burn-in time (Tburn): {Tburn} generations ({Tburn/Na:.1f}N)")
print(f"  Pop 1 size (N1): {N1}")
print(f"  Pop 2 size (N2): {N2}")
print(f"  Admixed pop size (NAdmix): {NAdmix}")
print(f"  Split time (Tsplit): {Tsplit} generations after burn-in")
print(f"  Admixture time (Tadmix): {Tadmix} generations after burn-in")
print(f"  End time (Tend): {Tend} generations after burn-in")
print(f"  Admixture fraction from pop1 (f): {f}")
print(f"  Genome length: {genome_length:.0e} bp")
print(f"  Recombination rate: {recomb_rate:.0e} per bp per gen")
print(f"  Mutation rate: {mutation_rate:.0e} per bp per gen")

## Generate SLiM5 Script

The following cell creates the SLiM5 script that implements the demographic model.

In [ ]:
# Create SLiM script
# Note: We do NOT add mutations during SLiM simulation
# Instead, we will use recapitation and add mutations afterward in tskit
slim_script = f"""
initialize() {{
    initializeTreeSeq();
    initializeMutationRate(0.0);  // No mutations during forward simulation
    initializeMutationType("m1", 0.5, "f", 0.0);  // Placeholder (required by SLiM)
    initializeGenomicElementType("g1", m1, 1.0);
    initializeGenomicElement(g1, 0, {int(genome_length)-1});
    initializeRecombinationRate({recomb_rate});
}}

// Create ancestral population
1 {{
    sim.addSubpop("p0", {Na});
}}

// Split into two populations at Tsplit
{Tsplit} {{
    sim.addSubpopSplit("p1", {N1}, p0);
    sim.addSubpopSplit("p2", {N2}, p0);
    p0.setSubpopulationSize(0);  // Remove ancestral population
}}

// Create admixed population at Tadmix
{Tadmix} {{
    sim.addSubpop("p3", {NAdmix});
    p3.setMigrationRates(c(p1, p2), c({f}, {1-f}));
}}

// Stop migration after one generation (admixture pulse)
{Tadmix + 1} {{
    p3.setMigrationRates(c(p1, p2), c(0.0, 0.0));
}}

// End simulation and output tree sequence
{Tend} late() {{
    sim.treeSeqOutput("{output_trees}");
    sim.simulationFinished();
}}
"""

# Save SLiM script to file
slim_script_file = "divergence_admixture.slim"
with open(slim_script_file, 'w') as f:
    f.write(slim_script)

print(f"SLiM script saved to: {slim_script_file}")
print("\nScript preview:")
print("=" * 60)
print(slim_script)
print("=" * 60)
print("\nNote: Mutations will be added after recapitation in tskit")

## Run SLiM Simulation

Execute the SLiM script to generate the tree sequence.

In [ ]:
# Run SLiM simulation
print("Running SLiM simulation...")
print("This may take a few moments depending on the parameter values.\n")

try:
    result = subprocess.run(
        ['slim', slim_script_file],
        capture_output=True,
        text=True,
        timeout=300  # 5 minute timeout
    )
    
    if result.returncode == 0:
        print("✓ Simulation completed successfully!")
        if result.stdout:
            print("\nSLiM output:")
            print(result.stdout)
    else:
        print("✗ Simulation failed!")
        print("Error output:")
        print(result.stderr)
        raise RuntimeError("SLiM simulation failed")
        
except FileNotFoundError:
    print("✗ Error: SLiM executable not found!")
    print("Please ensure SLiM is installed and in your PATH.")
    print("You can install SLiM from: https://messerlab.org/slim/")
    raise
except subprocess.TimeoutExpired:
    print("✗ Error: Simulation timed out!")
    print("Consider reducing population sizes or genome length.")
    raise

# Verify output file exists
if os.path.exists(output_trees):
    file_size = os.path.getsize(output_trees)
    print(f"\n✓ Tree sequence file created: {output_trees}")
    print(f"  File size: {file_size / 1024:.2f} KB")
else:
    print(f"\n✗ Error: Output file {output_trees} was not created!")
    raise FileNotFoundError(f"Expected output file {output_trees} not found")

## Load and Inspect Tree Sequence

Load the tree sequence using tskit and examine its basic properties.

In [ ]:
# Load tree sequence
ts = tskit.load(output_trees)

print("Tree Sequence Summary")
print("=" * 60)
print(f"Sequence length: {ts.sequence_length:,.0f} bp")
print(f"Number of trees: {ts.num_trees:,}")
print(f"Number of samples: {ts.num_samples:,}")
print(f"Number of nodes: {ts.num_nodes:,}")
print(f"Number of mutations: {ts.num_mutations:,}")
print(f"Number of populations: {ts.num_populations}")
print(f"Number of individuals: {ts.num_individuals:,}")

# Display population information
print("\nPopulation Information:")
print("-" * 60)
for pop in ts.populations():
    print(f"Population {pop.id}: {pop.metadata}")

# Count samples per population
print("\nSamples per population:")
print("-" * 60)
for pop_id in range(ts.num_populations):
    sample_nodes = [node.id for node in ts.nodes() if node.population == pop_id and node.is_sample()]
    print(f"Population {pop_id}: {len(sample_nodes)} samples")

## Recapitate and Add Mutations

To ensure the ancestral population has reached mutation-drift equilibrium, we use tskit's `recapitate()` function to simulate the coalescent history before the start of the forward simulation.

Then we add mutations explicitly using the `mutate()` function, which allows us to track when each mutation arose.

In [ ]:
# Recapitate to add coalescent history before the forward simulation
print("Recapitating tree sequence...")
print(f"  Adding {Tburn} generations of coalescent history")
print(f"  Ancestral population size: {Na}")

# Recapitate with appropriate parameters
ts_recap = tskit.recapitate(
    ts,
    recombination_rate=recomb_rate,
    population_size=Na,
    random_seed=42
)

print(f"\n✓ Recapitation complete")
print(f"  Original trees: {ts.num_trees:,}")
print(f"  After recapitation: {ts_recap.num_trees:,}")

# Add mutations to the tree sequence
print("\nAdding mutations...")
ts_mutated = tskit.mutate(
    ts_recap,
    rate=mutation_rate,
    random_seed=42,
    keep=False  # Don't keep existing mutations (there shouldn't be any)
)

print(f"\n✓ Mutations added")
print(f"  Number of mutations: {ts_mutated.num_mutations:,}")
print(f"  Number of sites: {ts_mutated.num_sites:,}")

# Store the split time in generations from the start (including recapitation)
split_time_total = Tsplit

print(f"\n✓ Tree sequence ready for analysis")
print(f"  Split occurred at generation {Tsplit} (forward time)")

# Update ts to use the mutated version for all downstream analyses
ts = ts_mutated

## Analyze Population Structure

Calculate genetic diversity statistics for each population.

In [ ]:
# Get sample nodes for each population
pop_samples = {}
for pop_id in range(ts.num_populations):
    pop_samples[pop_id] = [node.id for node in ts.nodes() if node.population == pop_id and node.is_sample()]

# Calculate diversity statistics
print("Genetic Diversity Statistics")
print("=" * 60)

diversity_stats = []

for pop_id, samples in pop_samples.items():
    if len(samples) >= 2:
        # Nucleotide diversity (π)
        pi = ts.diversity(sample_sets=[samples], mode='site')
        
        # Tajima's D
        try:
            tajd = ts.Tajimas_D(sample_sets=[samples], mode='site')
        except:
            tajd = np.nan
        
        diversity_stats.append({
            'Population': pop_id,
            'Samples': len(samples),
            'Pi': pi[0] if len(pi) > 0 else np.nan,
            'Tajimas_D': tajd[0] if hasattr(tajd, '__len__') and len(tajd) > 0 else tajd
        })
        
        print(f"Population {pop_id}:")
        print(f"  Samples: {len(samples)}")
        print(f"  Nucleotide diversity (π): {pi[0] if len(pi) > 0 else 'N/A'}")
        print(f"  Tajima's D: {tajd[0] if hasattr(tajd, '__len__') and len(tajd) > 0 else tajd}")
        print()

# Create DataFrame
if diversity_stats:
    df_diversity = pd.DataFrame(diversity_stats)
    print("\nSummary Table:")
    print(df_diversity.to_string(index=False))

## Calculate Pairwise Divergence (Dxy)

Calculate divergence between population pairs to examine the effects of split and admixture.

In [ ]:
# Calculate pairwise divergence between populations
print("Pairwise Divergence (Dxy) Between Populations")
print("=" * 60)

divergence_matrix = np.zeros((ts.num_populations, ts.num_populations))

for pop1_id in range(ts.num_populations):
    for pop2_id in range(pop1_id, ts.num_populations):
        samples1 = pop_samples.get(pop1_id, [])
        samples2 = pop_samples.get(pop2_id, [])
        
        if len(samples1) >= 1 and len(samples2) >= 1:
            if pop1_id == pop2_id:
                # Within-population diversity
                dxy = ts.diversity(sample_sets=[samples1], mode='site')
            else:
                # Between-population divergence
                dxy = ts.divergence(sample_sets=[samples1, samples2], mode='site')
            
            div_value = dxy[0] if len(dxy) > 0 else 0
            divergence_matrix[pop1_id, pop2_id] = div_value
            divergence_matrix[pop2_id, pop1_id] = div_value
            
            if pop1_id != pop2_id:
                print(f"Pop {pop1_id} vs Pop {pop2_id}: {div_value:.6f}")

# Visualize divergence matrix
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(divergence_matrix, cmap='YlOrRd', aspect='auto')
ax.set_xticks(range(ts.num_populations))
ax.set_yticks(range(ts.num_populations))
ax.set_xlabel('Population')
ax.set_ylabel('Population')
ax.set_title('Pairwise Genetic Divergence (Dxy)')
plt.colorbar(im, ax=ax, label='Divergence')

# Add text annotations
for i in range(ts.num_populations):
    for j in range(ts.num_populations):
        text = ax.text(j, i, f'{divergence_matrix[i, j]:.4f}',
                      ha="center", va="center", color="black", fontsize=10)

plt.tight_layout()
plt.show()

## Calculate FST Between Populations

FST measures population differentiation and helps quantify the genetic distance between populations.

In [ ]:
# Calculate FST between population pairs
print("Pairwise FST Between Populations")
print("=" * 60)

fst_matrix = np.zeros((ts.num_populations, ts.num_populations))

for pop1_id in range(ts.num_populations):
    for pop2_id in range(pop1_id + 1, ts.num_populations):
        samples1 = pop_samples.get(pop1_id, [])
        samples2 = pop_samples.get(pop2_id, [])
        
        if len(samples1) >= 2 and len(samples2) >= 2:
            fst = ts.Fst(sample_sets=[samples1, samples2], mode='site')
            fst_value = fst[0] if len(fst) > 0 else 0
            fst_matrix[pop1_id, pop2_id] = fst_value
            fst_matrix[pop2_id, pop1_id] = fst_value
            
            print(f"Pop {pop1_id} vs Pop {pop2_id}: FST = {fst_value:.6f}")

# Visualize FST matrix
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(fst_matrix, cmap='viridis', aspect='auto', vmin=0, vmax=1)
ax.set_xticks(range(ts.num_populations))
ax.set_yticks(range(ts.num_populations))
ax.set_xlabel('Population')
ax.set_ylabel('Population')
ax.set_title('Pairwise FST Between Populations')
plt.colorbar(im, ax=ax, label='FST')

# Add text annotations
for i in range(ts.num_populations):
    for j in range(ts.num_populations):
        if i != j:
            text = ax.text(j, i, f'{fst_matrix[i, j]:.4f}',
                          ha="center", va="center", color="white", fontsize=10)

plt.tight_layout()
plt.show()

## Visualize Site Frequency Spectrum

The site frequency spectrum (SFS) shows the distribution of allele frequencies in each population.

In [ ]:
# Calculate and plot site frequency spectrum for each population
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for pop_id, samples in pop_samples.items():
    if len(samples) >= 2 and pop_id < 4:
        # Calculate allele frequency spectrum
        afs = ts.allele_frequency_spectrum(
            sample_sets=[samples],
            mode='site',
            polarised=True
        )
        
        # Plot SFS (excluding monomorphic sites)
        ax = axes[pop_id]
        x = np.arange(1, len(afs))  # Exclude 0 frequency
        y = afs[1:]  # Exclude monomorphic sites
        
        ax.bar(x, y, color=f'C{pop_id}', alpha=0.7)
        ax.set_xlabel('Derived Allele Count')
        ax.set_ylabel('Number of Sites')
        ax.set_title(f'Site Frequency Spectrum - Population {pop_id}\n({len(samples)} samples)')
        ax.set_yscale('log')
        ax.grid(True, alpha=0.3)

# Remove extra subplot if fewer than 4 populations
for i in range(len(pop_samples), 4):
    axes[i].axis('off')

plt.tight_layout()
plt.show()

## Visualize Tree Sequence

Draw a representation of the first few trees in the sequence to visualize genealogical relationships.

In [ ]:
# Simplify tree sequence for visualization (sample a subset if needed)
# Take up to 20 samples per population for clearer visualization
viz_samples = []
for pop_id, samples in pop_samples.items():
    viz_samples.extend(samples[:min(20, len(samples))])

# Draw the first tree
if ts.num_trees > 0:
    tree = ts.first()
    print(f"Visualizing first tree (interval: {tree.interval})")
    print(f"Total trees in sequence: {ts.num_trees}")
    
    # Create a color map for populations
    node_colors = {}
    color_map = {0: 'red', 1: 'blue', 2: 'green', 3: 'purple'}
    
    for node in ts.nodes():
        if node.population in color_map:
            node_colors[node.id] = color_map[node.population]
    
    # Note: Tree visualization in Jupyter requires display capability
    # For basic text representation:
    print("\nTree structure (text representation):")
    print(tree.draw_text())

## Dxy Decomposition by Mutation Origin

This analysis decomposes the genetic divergence (Dxy) between populations 1 and 2 into two components:
1. **Ancestral polymorphism**: Mutations that existed in the ancestral population before the split
2. **New mutations**: Mutations that arose independently in each population after the split

We identify mutation origins by examining the time when each mutation arose relative to the split time.

In [ ]:
# Dxy decomposition analysis
print("Analyzing Dxy decomposition by mutation origin")
print("=" * 60)

# Get samples from populations 1 and 2
pop1_samples = pop_samples.get(1, [])
pop2_samples = pop_samples.get(2, [])

if len(pop1_samples) == 0 or len(pop2_samples) == 0:
    print("Warning: Need samples from both population 1 and 2")
else:
    # Calculate total Dxy between populations
    dxy_total = ts.divergence(sample_sets=[pop1_samples, pop2_samples], mode='site')
    dxy_total_value = dxy_total[0] if len(dxy_total) > 0 else 0
    
    print(f"Total Dxy between pop1 and pop2: {dxy_total_value:.6f}")
    print(f"Split time: {Tsplit} generations (forward time)")
    
    # Classify mutations by origin
    ancestral_mutations = []
    new_mutations = []
    
    # Iterate through all mutations
    for mut in ts.mutations():
        # Get the time of the mutation (in generations ago from present)
        # For tree sequences, time 0 is present, and time increases going back
        node = ts.node(mut.node)
        mut_time = node.time
        
        # The split occurred at generation Tsplit in forward time
        # In time-ago from present: split_time_ago = Tend - Tsplit
        split_time_ago = Tend - Tsplit
        
        # If mutation time is greater than split time, it's ancestral
        if mut_time > split_time_ago:
            ancestral_mutations.append(mut)
        else:
            new_mutations.append(mut)
    
    
    total_mutations = ts.num_mutations
    print(f"\nMutation classification:")
    print(f"  Ancestral (pre-split): {len(ancestral_mutations):,} mutations")
    print(f"  New (post-split): {len(new_mutations):,} mutations")
    print(f"  Total: {total_mutations:,} mutations")
    
    # Calculate Dxy contribution from each class
    # We need to count pairwise differences at sites with each class of mutations
    
    ancestral_dxy = 0
    new_dxy = 0
    
    # Create sets of site IDs for each mutation class
    ancestral_sites = set(mut.site for mut in ancestral_mutations)
    new_sites = set(mut.site for mut in new_mutations)
    
    # Calculate divergence for each site
    for site in ts.sites():
        # Get genotypes at this site
        genotypes = []
        for sample in pop1_samples + pop2_samples:
            # Find genotype for this sample at this site
            gt = 0
            for mut in site.mutations:
                # Check if this sample carries the mutation
                node = mut.node
                # Simple check: is the sample descended from this node?
                # This is simplified; proper implementation would trace tree
            genotypes.append(gt)
    
    # Alternative approach: use branch lengths
    # Calculate divergence using only ancestral polymorphisms
    print(f"\nDxy decomposition (approximate):")
    prop_ancestral = len(ancestral_mutations) / max(total_mutations, 1)
    prop_new = len(new_mutations) / max(total_mutations, 1)
    
    dxy_ancestral_est = dxy_total_value * prop_ancestral
    dxy_new_est = dxy_total_value * prop_new
    
    print(f"  Ancestral polymorphism contribution: {dxy_ancestral_est:.6f} ({prop_ancestral*100:.1f}%)")
    print(f"  New mutation contribution: {dxy_new_est:.6f} ({prop_new*100:.1f}%)")
    
    # Store results for plotting
    dxy_results = {
        'total': dxy_total_value,
        'ancestral': dxy_ancestral_est,
        'new': dxy_new_est,
        'split_time': Tsplit,
        'n_ancestral_muts': len(ancestral_mutations),
        'n_new_muts': len(new_mutations)
    }

## Visualize Dxy Decomposition

Create a figure showing the decomposition of Dxy into ancestral and new mutation components.

In [ ]:
# Improved Dxy decomposition calculation
print("Computing precise Dxy decomposition...")

if len(pop1_samples) > 0 and len(pop2_samples) > 0:
    # Get genotype matrix
    genotypes = ts.genotype_matrix()
    
    # Calculate split time in "time ago" coordinates
    split_time_ago = Tend - Tsplit
    
    # Classify each site by mutation origin
    ancestral_divergence = 0
    new_divergence = 0
    total_comparisons = len(pop1_samples) * len(pop2_samples)
    
    for site_idx, site in enumerate(ts.sites()):
        # Determine if this site has ancestral or new mutations
        is_ancestral = False
        for mut in site.mutations:
            node = ts.node(mut.node)
            if node.time > split_time_ago:
                is_ancestral = True
                break
        
        # Count pairwise differences at this site
        site_divergence = 0
        for i in pop1_samples:
            for j in pop2_samples:
                if genotypes[site_idx, i] != genotypes[site_idx, j]:
                    site_divergence += 1
        
        site_dxy = site_divergence / total_comparisons
        
        if is_ancestral:
            ancestral_divergence += site_dxy
        else:
            new_divergence += site_dxy
    
    # Normalize by sequence length
    ancestral_dxy = ancestral_divergence / ts.sequence_length
    new_dxy = new_divergence / ts.sequence_length
    total_dxy = ancestral_dxy + new_dxy
    
    print(f"\nPrecise Dxy decomposition:")
    print(f"  Total Dxy: {total_dxy:.6f}")
    print(f"  Ancestral contribution: {ancestral_dxy:.6f} ({ancestral_dxy/max(total_dxy, 1e-10)*100:.1f}%)")
    print(f"  New mutation contribution: {new_dxy:.6f} ({new_dxy/max(total_dxy, 1e-10)*100:.1f}%)")
    
    # Create visualization
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Panel 1: Bar plot of contributions
    ax = axes[0]
    categories = ['Ancestral\nPolymorphism', 'New\nMutations', 'Total']
    values = [ancestral_dxy, new_dxy, total_dxy]
    colors = ['#3498db', '#e74c3c', '#2ecc71']
    
    bars = ax.bar(categories, values, color=colors, alpha=0.7, edgecolor='black', linewidth=1.5)
    ax.set_ylabel('Dxy', fontsize=12, fontweight='bold')
    ax.set_title(f'Dxy Decomposition\n(Split time = {Tsplit} generations)', 
                 fontsize=13, fontweight='bold')
    ax.grid(axis='y', alpha=0.3, linestyle='--')
    
    # Add value labels on bars
    for bar, val in zip(bars, values):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{val:.4f}',
                ha='center', va='bottom', fontsize=10, fontweight='bold')
    
    # Panel 2: Stacked bar showing proportions
    ax = axes[1]
    prop_ancestral = ancestral_dxy / max(total_dxy, 1e-10)
    prop_new = new_dxy / max(total_dxy, 1e-10)
    
    ax.bar(['Dxy Components'], [prop_ancestral], label='Ancestral', 
           color='#3498db', alpha=0.7, edgecolor='black', linewidth=1.5)
    ax.bar(['Dxy Components'], [prop_new], bottom=[prop_ancestral], 
           label='New mutations', color='#e74c3c', alpha=0.7, 
           edgecolor='black', linewidth=1.5)
    
    ax.set_ylabel('Proportion of Dxy', fontsize=12, fontweight='bold')
    ax.set_title('Relative Contributions to Dxy', fontsize=13, fontweight='bold')
    ax.set_ylim([0, 1])
    ax.legend(loc='upper right', fontsize=11)
    ax.grid(axis='y', alpha=0.3, linestyle='--')
    
    # Add percentage labels
    ax.text(0, prop_ancestral/2, f'{prop_ancestral*100:.1f}%', 
            ha='center', va='center', fontsize=12, fontweight='bold', color='white')
    ax.text(0, prop_ancestral + prop_new/2, f'{prop_new*100:.1f}%', 
            ha='center', va='center', fontsize=12, fontweight='bold', color='white')
    
    plt.tight_layout()
    plt.show()
    
    # Store results
    dxy_decomp_results = {
        'split_time': Tsplit,
        'total_dxy': total_dxy,
        'ancestral_dxy': ancestral_dxy,
        'new_dxy': new_dxy,
        'prop_ancestral': prop_ancestral,
        'prop_new': prop_new
    }
    
    print("\n✓ Visualization complete")
else:
    print("Cannot create visualization: need samples from both populations")

## Dxy vs Split Time Analysis

To understand how the contributions to Dxy change with split time, we can run multiple simulations with different split times and track the ancestral vs. new mutation contributions.

**Note**: This analysis requires running multiple simulations, which can be time-consuming. The cell below provides a framework that can be run with different parameter values.

In [ ]:
# Analysis framework: Dxy as a function of split time
# This demonstrates how to run the analysis for multiple split times

print("Dxy vs Split Time Analysis Framework")
print("=" * 60)
print("\nTo run a complete sweep:")
print("1. Define a range of split times")
print("2. For each split time:")
print("   - Generate new SLiM script")
print("   - Run simulation")
print("   - Recapitate and add mutations")
print("   - Calculate Dxy decomposition")
print("3. Plot results\n")

# Example: Store current results for comparison
if 'dxy_decomp_results' in locals():
    split_time_results = [{
        'split_time': dxy_decomp_results['split_time'],
        'total_dxy': dxy_decomp_results['total_dxy'],
        'ancestral_dxy': dxy_decomp_results['ancestral_dxy'],
        'new_dxy': dxy_decomp_results['new_dxy']
    }]
    
    print(f"Current simulation results:")
    print(f"  Split time: {split_time_results[0]['split_time']} generations")
    print(f"  Total Dxy: {split_time_results[0]['total_dxy']:.6f}")
    print(f"  Ancestral: {split_time_results[0]['ancestral_dxy']:.6f}")
    print(f"  New: {split_time_results[0]['new_dxy']:.6f}")
    
    # Theoretical expectation
    print(f"\nTheoretical expectations:")
    print(f"  - Ancestral contribution should remain relatively constant")
    print(f"  - New mutation contribution should increase linearly with split time")
    print(f"  - Total Dxy = Ancestral + New ≈ constant + 2*μ*T_split")
    print(f"    where μ = {mutation_rate:.2e} and T_split = {Tsplit}")
    print(f"    Expected new contribution ≈ 2*{mutation_rate:.2e}*{Tsplit} = {2*mutation_rate*Tsplit:.6f}")
else:
    print("Run previous cells first to get decomposition results")

## Run Multiple Simulations with Different Split Times

This cell runs a parameter sweep over different split times to show how Dxy components change.

**Warning**: This will run multiple SLiM simulations and may take several minutes.

In [ ]:
# Run multiple simulations with different split times
import os
import tempfile

# Define split times to test (in generations)
split_times = [500, 1000, 1500, 2000, 2500]
run_sweep = False  # Set to True to run time-consuming sweep analysis

if run_sweep:
    print("Running split time sweep...")
    print(f"Testing {len(split_times)} different split times")
    print("This may take several minutes...\n")
    
    sweep_results = []
    
    for split_t in split_times:
        print(f"Processing split time = {split_t} generations...")
        
        # Create temporary output file
        temp_output = f"temp_split_{split_t}.trees"
        
        # Generate SLiM script for this split time
        slim_script_temp = f"""
initialize() {{
    initializeTreeSeq();
    initializeMutationRate(0.0);
    initializeMutationType("m1", 0.5, "f", 0.0);
    initializeGenomicElementType("g1", m1, 1.0);
    initializeGenomicElement(g1, 0, {int(genome_length)-1});
    initializeRecombinationRate({recomb_rate});
}}

1 {{
    sim.addSubpop("p0", {Na});
}}

{split_t} {{
    sim.addSubpopSplit("p1", {N1}, p0);
    sim.addSubpopSplit("p2", {N2}, p0);
    p0.setSubpopulationSize(0);
}}

{Tadmix} late() {{
    sim.treeSeqOutput("{temp_output}");
    sim.simulationFinished();
}}
"""
        
        # Save and run script
        temp_slim = f"temp_split_{split_t}.slim"
        with open(temp_slim, 'w') as f:
            f.write(slim_script_temp)
        
        try:
            # Run SLiM
            result = subprocess.run(
                ['slim', temp_slim],
                capture_output=True,
                text=True,
                timeout=120
            )
            
            if result.returncode == 0 and os.path.exists(temp_output):
                # Load tree sequence
                ts_temp = tskit.load(temp_output)
                
                # Recapitate
                ts_temp = tskit.recapitate(
                    ts_temp,
                    recombination_rate=recomb_rate,
                    population_size=Na,
                    random_seed=42
                )
                
                # Add mutations
                ts_temp = tskit.mutate(ts_temp, rate=mutation_rate, random_seed=42)
                
                # Get samples
                temp_pop_samples = {}
                for pop_id in range(ts_temp.num_populations):
                    temp_pop_samples[pop_id] = [
                        node.id for node in ts_temp.nodes() 
                        if node.population == pop_id and node.is_sample()
                    ]
                
                temp_pop1 = temp_pop_samples.get(1, [])
                temp_pop2 = temp_pop_samples.get(2, [])
                
                if len(temp_pop1) > 0 and len(temp_pop2) > 0:
                    # Calculate Dxy decomposition
                    genotypes = ts_temp.genotype_matrix()
                    split_time_ago = Tadmix - split_t
                    
                    ancestral_div = 0
                    new_div = 0
                    total_comps = len(temp_pop1) * len(temp_pop2)
                    
                    for site_idx, site in enumerate(ts_temp.sites()):
                        is_ancestral = any(
                            ts_temp.node(mut.node).time > split_time_ago 
                            for mut in site.mutations
                        )
                        
                        site_div = sum(
                            genotypes[site_idx, i] != genotypes[site_idx, j]
                            for i in temp_pop1 for j in temp_pop2
                        )
                        
                        site_dxy = site_div / total_comps
                        
                        if is_ancestral:
                            ancestral_div += site_dxy
                        else:
                            new_div += site_dxy
                    
                    ancestral_dxy = ancestral_div / ts_temp.sequence_length
                    new_dxy = new_div / ts_temp.sequence_length
                    total_dxy = ancestral_dxy + new_dxy
                    
                    sweep_results.append({
                        'split_time': split_t,
                        'total_dxy': total_dxy,
                        'ancestral_dxy': ancestral_dxy,
                        'new_dxy': new_dxy
                    })
                    
                    print(f"  ✓ Total Dxy: {total_dxy:.6f} (Ancestral: {ancestral_dxy:.6f}, New: {new_dxy:.6f})")
                
                # Clean up temporary files
                os.remove(temp_output)
            else:
                print(f"  ✗ Simulation failed for split time {split_t}")
            
            os.remove(temp_slim)
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
            continue
    
    print(f"\n✓ Completed {len(sweep_results)} simulations")
    
else:
    run_sweep = False  # Set to True to run time-consuming sweep analysis
    # Use mock data for demonstration
    sweep_results = [
        {'split_time': 500, 'total_dxy': 0.000150, 'ancestral_dxy': 0.000040, 'new_dxy': 0.000110},
        {'split_time': 1000, 'total_dxy': 0.000240, 'ancestral_dxy': 0.000040, 'new_dxy': 0.000200},
        {'split_time': 1500, 'total_dxy': 0.000340, 'ancestral_dxy': 0.000040, 'new_dxy': 0.000300},
        {'split_time': 2000, 'total_dxy': 0.000440, 'ancestral_dxy': 0.000040, 'new_dxy': 0.000400},
        {'split_time': 2500, 'total_dxy': 0.000540, 'ancestral_dxy': 0.000040, 'new_dxy': 0.000500},
    ]
    print("Using mock data for demonstration")

## Visualize Dxy vs Split Time

This plot shows how total Dxy and its components (ancestral polymorphism vs. new mutations) change as a function of split time.

In [ ]:
# Create comprehensive visualization of Dxy vs split time
if 'sweep_results' in locals() and len(sweep_results) > 0:
    # Extract data
    split_times_data = [r['split_time'] for r in sweep_results]
    total_dxy_data = [r['total_dxy'] for r in sweep_results]
    ancestral_dxy_data = [r['ancestral_dxy'] for r in sweep_results]
    new_dxy_data = [r['new_dxy'] for r in sweep_results]
    
    # Create figure with multiple panels
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Panel 1: Dxy components vs split time
    ax = axes[0, 0]
    ax.plot(split_times_data, total_dxy_data, 'o-', linewidth=2.5, markersize=8,
            label='Total Dxy', color='#2ecc71', markeredgecolor='black', markeredgewidth=1)
    ax.plot(split_times_data, ancestral_dxy_data, 's-', linewidth=2, markersize=7,
            label='Ancestral polymorphism', color='#3498db', markeredgecolor='black', markeredgewidth=1)
    ax.plot(split_times_data, new_dxy_data, '^-', linewidth=2, markersize=7,
            label='New mutations', color='#e74c3c', markeredgecolor='black', markeredgewidth=1)
    
    ax.set_xlabel('Split Time (generations)', fontsize=12, fontweight='bold')
    ax.set_ylabel('Dxy', fontsize=12, fontweight='bold')
    ax.set_title('Dxy Components vs Split Time', fontsize=13, fontweight='bold')
    ax.legend(loc='upper left', fontsize=10, framealpha=0.9)
    ax.grid(True, alpha=0.3, linestyle='--')
    
    # Panel 2: Stacked area plot
    ax = axes[0, 1]
    ax.fill_between(split_times_data, 0, ancestral_dxy_data, 
                     label='Ancestral', color='#3498db', alpha=0.6)
    ax.fill_between(split_times_data, ancestral_dxy_data, total_dxy_data,
                     label='New mutations', color='#e74c3c', alpha=0.6)
    ax.plot(split_times_data, total_dxy_data, 'k-', linewidth=2, label='Total')
    
    ax.set_xlabel('Split Time (generations)', fontsize=12, fontweight='bold')
    ax.set_ylabel('Dxy', fontsize=12, fontweight='bold')
    ax.set_title('Stacked Dxy Components', fontsize=13, fontweight='bold')
    ax.legend(loc='upper left', fontsize=10, framealpha=0.9)
    ax.grid(True, alpha=0.3, linestyle='--')
    
    # Panel 3: Proportion of Dxy from each source
    ax = axes[1, 0]
    prop_ancestral = [a/max(t, 1e-10) for a, t in zip(ancestral_dxy_data, total_dxy_data)]
    prop_new = [n/max(t, 1e-10) for n, t in zip(new_dxy_data, total_dxy_data)]
    
    ax.plot(split_times_data, prop_ancestral, 's-', linewidth=2, markersize=7,
            label='Ancestral proportion', color='#3498db', markeredgecolor='black', markeredgewidth=1)
    ax.plot(split_times_data, prop_new, '^-', linewidth=2, markersize=7,
            label='New mutation proportion', color='#e74c3c', markeredgecolor='black', markeredgewidth=1)
    
    ax.set_xlabel('Split Time (generations)', fontsize=12, fontweight='bold')
    ax.set_ylabel('Proportion of Total Dxy', fontsize=12, fontweight='bold')
    ax.set_title('Relative Contributions to Dxy', fontsize=13, fontweight='bold')
    ax.set_ylim([0, 1])
    ax.legend(loc='center right', fontsize=10, framealpha=0.9)
    ax.grid(True, alpha=0.3, linestyle='--')
    
    # Panel 4: Rate of Dxy accumulation
    ax = axes[1, 1]
    if len(split_times_data) > 1:
        # Calculate rate of change
        rates_total = []
        rates_ancestral = []
        rates_new = []
        time_points = []
        
        for i in range(1, len(split_times_data)):
            dt = split_times_data[i] - split_times_data[i-1]
            if dt > 0:
                rate_t = (total_dxy_data[i] - total_dxy_data[i-1]) / dt
                rate_a = (ancestral_dxy_data[i] - ancestral_dxy_data[i-1]) / dt
                rate_n = (new_dxy_data[i] - new_dxy_data[i-1]) / dt
                
                rates_total.append(rate_t)
                rates_ancestral.append(rate_a)
                rates_new.append(rate_n)
                time_points.append((split_times_data[i] + split_times_data[i-1]) / 2)
        
        ax.plot(time_points, rates_total, 'o-', linewidth=2, markersize=7,
                label='Total', color='#2ecc71', markeredgecolor='black', markeredgewidth=1)
        ax.plot(time_points, rates_new, '^-', linewidth=2, markersize=7,
                label='New mutations', color='#e74c3c', markeredgecolor='black', markeredgewidth=1)
        
        # Add theoretical expectation line
        expected_rate = 2 * mutation_rate
        ax.axhline(y=expected_rate, color='black', linestyle='--', linewidth=2,
                   label=f'Expected: 2μ = {expected_rate:.2e}')
        
        ax.set_xlabel('Split Time (generations)', fontsize=12, fontweight='bold')
        ax.set_ylabel('dDxy/dt (per generation)', fontsize=12, fontweight='bold')
        ax.set_title('Rate of Dxy Accumulation', fontsize=13, fontweight='bold')
        ax.legend(loc='best', fontsize=9, framealpha=0.9)
        ax.grid(True, alpha=0.3, linestyle='--')
        ax.ticklabel_format(axis='y', style='scientific', scilimits=(0,0))
    
    plt.tight_layout()
    plt.show()
    
    # Print summary statistics
    print("\nSummary Statistics:")
    print("=" * 60)
    print(f"Number of simulations: {len(sweep_results)}")
    print(f"Split time range: {min(split_times_data)} - {max(split_times_data)} generations")
    print(f"\nDxy range:")
    print(f"  Total: {min(total_dxy_data):.6f} - {max(total_dxy_data):.6f}")
    print(f"  Ancestral: {min(ancestral_dxy_data):.6f} - {max(ancestral_dxy_data):.6f}")
    print(f"  New: {min(new_dxy_data):.6f} - {max(new_dxy_data):.6f}")
    print(f"\nTheoretical predictions:")
    print(f"  Expected Dxy accumulation rate: 2μ = {2*mutation_rate:.2e} per generation")
    print(f"  For split time T, expected new contribution ≈ 2μT")
    
    # Compare first and last simulations
    if len(sweep_results) >= 2:
        first = sweep_results[0]
        last = sweep_results[-1]
        print(f"\nComparison (first vs last):")
        print(f"  Split time increased by: {last['split_time'] - first['split_time']} generations")
        print(f"  Total Dxy increased by: {last['total_dxy'] - first['total_dxy']:.6f}")
        print(f"  Ancestral Dxy changed by: {last['ancestral_dxy'] - first['ancestral_dxy']:.6f}")
        print(f"  New Dxy increased by: {last['new_dxy'] - first['new_dxy']:.6f}")
        
        observed_rate = (last['new_dxy'] - first['new_dxy']) / (last['split_time'] - first['split_time'])
        expected_rate = 2 * mutation_rate
        print(f"\n  Observed accumulation rate: {observed_rate:.2e} per generation")
        print(f"  Expected accumulation rate: {expected_rate:.2e} per generation")
        print(f"  Ratio (observed/expected): {observed_rate/expected_rate:.2f}")
    
else:
    print("No sweep results available. Run the previous cell first.")

## Analyze Admixture Proportions

Examine the genetic ancestry of the admixed population by looking at the proportion of ancestry from each source population.

In [ ]:
# Analyze ancestry proportions in the admixed population (p3)
if 3 in pop_samples and len(pop_samples[3]) > 0:
    print("Admixture Analysis")
    print("=" * 60)
    
    admixed_samples = pop_samples[3]
    pop1_samples = pop_samples.get(1, [])
    pop2_samples = pop_samples.get(2, [])
    
    if len(pop1_samples) > 0 and len(pop2_samples) > 0:
        # Calculate genetic similarity of admixed population to each source
        # Using divergence as a proxy for ancestry
        
        dxy_admix_p1 = ts.divergence(sample_sets=[admixed_samples, pop1_samples], mode='site')
        dxy_admix_p2 = ts.divergence(sample_sets=[admixed_samples, pop2_samples], mode='site')
        dxy_p1_p2 = ts.divergence(sample_sets=[pop1_samples, pop2_samples], mode='site')
        
        div_admix_p1 = dxy_admix_p1[0] if len(dxy_admix_p1) > 0 else 0
        div_admix_p2 = dxy_admix_p2[0] if len(dxy_admix_p2) > 0 else 0
        div_p1_p2 = dxy_p1_p2[0] if len(dxy_p1_p2) > 0 else 0
        
        print(f"Divergence between admixed pop and Pop1: {div_admix_p1:.6f}")
        print(f"Divergence between admixed pop and Pop2: {div_admix_p2:.6f}")
        print(f"Divergence between Pop1 and Pop2: {div_p1_p2:.6f}")
        
        # Estimate admixture proportion using divergence
        # Expected: div(admix, p1) ≈ (1-f) * div(p1, p2)
        # Expected: div(admix, p2) ≈ f * div(p1, p2)
        if div_p1_p2 > 0:
            est_f = 1 - (div_admix_p1 / div_p1_p2)
            print(f"\nExpected admixture fraction from Pop1: {f:.3f}")
            print(f"Estimated admixture fraction from Pop1: {est_f:.3f}")
            
            # Visualize
            fig, ax = plt.subplots(figsize=(8, 5))
            categories = ['Expected', 'Estimated']
            values = [f, est_f]
            colors = ['lightblue', 'orange']
            
            ax.bar(categories, values, color=colors, alpha=0.7, edgecolor='black')
            ax.set_ylabel('Admixture Fraction from Pop1')
            ax.set_title('Admixture Proportion Comparison')
            ax.set_ylim([0, 1])
            ax.axhline(y=f, color='red', linestyle='--', alpha=0.5, label=f'Expected (f={f})')
            ax.legend()
            
            # Add value labels on bars
            for i, v in enumerate(values):
                ax.text(i, v + 0.02, f'{v:.3f}', ha='center', va='bottom', fontsize=12, fontweight='bold')
            
            plt.tight_layout()
            plt.show()
else:
    print("No admixed population samples found for analysis.")

## Local Ancestry Visualization

Visualize how hybridization creates a mosaic of recombined ancestries along the genome.

This analysis shows local ancestry patterns in:
- **F1 hybrids**: First-generation crosses showing 50-50 ancestry blocks
- **F2 hybrids**: Second-generation showing recombination between ancestries
- **Backcrosses**: First-generation backcross to one parental population
- **Late-generation hybrids**: Multiple generations of recombination

Local ancestry is tracked by examining which source population each genomic segment was inherited from, based on the tree sequence genealogy.

In [ ]:
# Function to track local ancestry along the genome
def get_local_ancestry(ts, sample_node, pop1_id=1, pop2_id=2, num_windows=100):
    """
    Track local ancestry for a sample along the genome.
    
    For each genomic window, traces the genealogy back to determine
    which source population the ancestry comes from.
    
    Returns:
        windows: genomic positions
        ancestry: array where 1 = pop1, 2 = pop2, 0 = uncertain/ancestral
    """
    sequence_length = ts.sequence_length
    window_size = sequence_length / num_windows
    windows = np.linspace(0, sequence_length, num_windows + 1)
    ancestry = np.zeros(num_windows)
    
    # Sample points within each window
    for i in range(num_windows):
        pos = (windows[i] + windows[i+1]) / 2
        
        # Find the tree at this position
        tree = ts.at(pos)
        
        # Trace ancestry from sample node back in time
        node = sample_node
        
        # Follow the tree upward until we find a node in one of the source populations
        # We need to identify which population at the time of admixture
        while node != tskit.NULL:
            node_obj = ts.node(node)
            node_pop = node_obj.population
            node_time = node_obj.time
            
            # Check if this node is in one of our source populations
            # We're looking for nodes that existed at or before the admixture time
            if node_pop == pop1_id:
                ancestry[i] = 1
                break
            elif node_pop == pop2_id:
                ancestry[i] = 2
                break
            
            # Move to parent
            parent = tree.parent(node)
            if parent == tskit.NULL:
                # Reached root without finding source population
                ancestry[i] = 0  # Ancestral/uncertain
                break
            node = parent
    
    return windows, ancestry

print("✓ Local ancestry tracking function defined")

In [ ]:
# Function to visualize local ancestry as colored blocks
def plot_local_ancestry(windows, ancestry_data, ax, title, sample_labels=None):
    """
    Plot local ancestry patterns as colored horizontal bars.
    
    Parameters:
    -----------
    windows : array
        Genomic positions
    ancestry_data : list of arrays
        List of ancestry arrays (one per sample)
    ax : matplotlib axis
        Axis to plot on
    title : str
        Title for the plot
    sample_labels : list of str, optional
        Labels for each sample
    """
    n_samples = len(ancestry_data)
    
    # Colors for different ancestries
    colors = {
        0: '#CCCCCC',  # Gray for uncertain/ancestral
        1: '#3498db',  # Blue for population 1
        2: '#e74c3c'   # Red for population 2
    }
    
    # Plot each sample as a horizontal bar
    for sample_idx, ancestry in enumerate(ancestry_data):
        y_pos = n_samples - sample_idx - 1  # Top to bottom
        
        # Plot each window as a colored rectangle
        for i in range(len(ancestry)):
            x_start = windows[i]
            x_end = windows[i + 1]
            width = x_end - x_start
            
            anc = int(ancestry[i])
            color = colors.get(anc, '#CCCCCC')
            
            ax.add_patch(plt.Rectangle(
                (x_start, y_pos - 0.4), width, 0.8,
                facecolor=color, edgecolor='none'
            ))
    
    # Formatting
    ax.set_xlim(0, windows[-1])
    ax.set_ylim(-0.5, n_samples - 0.5)
    ax.set_xlabel('Genomic Position (bp)', fontsize=11, fontweight='bold')
    ax.set_ylabel('Individual', fontsize=11, fontweight='bold')
    ax.set_title(title, fontsize=12, fontweight='bold')
    
    # Y-axis labels
    if sample_labels:
        ax.set_yticks(range(n_samples))
        ax.set_yticklabels([sample_labels[i] for i in range(n_samples)][::-1])
    else:
        ax.set_yticks(range(n_samples))
        ax.set_yticklabels([f'Sample {i+1}' for i in range(n_samples)][::-1])
    
    # Add legend
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor=colors[1], label='Population 1'),
        Patch(facecolor=colors[2], label='Population 2'),
        Patch(facecolor=colors[0], label='Ancestral')
    ]
    ax.legend(handles=legend_elements, loc='upper right', fontsize=9)
    
    ax.grid(axis='x', alpha=0.3, linestyle='--')

print("✓ Visualization function defined")

## Visualize Ancestry Mosaics for Different Hybrid Generations

We'll create ancestry plots for samples from the admixed population (p3) to show:

### Expected Patterns

1. **F1 hybrids**: Should show large blocks of ancestry, roughly 50% from each parent population
2. **F2 hybrids**: Second generation shows more recombination breakpoints, creating smaller ancestry blocks
3. **Backcrosses**: Asymmetric ancestry (e.g., 75% from one parent, 25% from the other)
4. **Late-generation hybrids**: Fine-scale mosaic with many small ancestry blocks due to accumulated recombination

The visualization uses colors to distinguish ancestries:
- **Blue**: Ancestry from Population 1
- **Red**: Ancestry from Population 2
- **Gray**: Ancestral (pre-split) or uncertain

In [ ]:
# Analyze local ancestry for admixed population samples
if 3 in pop_samples and len(pop_samples[3]) > 0:
    print("Analyzing local ancestry in admixed population...")
    print("=" * 60)
    
    admixed_samples = pop_samples[3]
    
    # Select samples to visualize (up to 8 for clarity)
    n_samples_to_plot = min(8, len(admixed_samples))
    selected_samples = admixed_samples[:n_samples_to_plot]
    
    print(f"Analyzing {n_samples_to_plot} samples from admixed population")
    print(f"Total admixed samples available: {len(admixed_samples)}")
    print(f"Genome length: {ts.sequence_length:,.0f} bp\n")
    
    # Track ancestry for each sample
    ancestry_results = []
    windows = None
    
    for i, sample_node in enumerate(selected_samples):
        print(f"Processing sample {i+1}/{n_samples_to_plot}...", end='\r')
        windows, ancestry = get_local_ancestry(ts, sample_node, pop1_id=1, pop2_id=2, num_windows=200)
        ancestry_results.append(ancestry)
    
    print(f"✓ Processed {n_samples_to_plot} samples" + " " * 20)
    
    # Calculate ancestry statistics
    print("\nAncestry Statistics:")
    print("-" * 60)
    for i, ancestry in enumerate(ancestry_results):
        prop_pop1 = np.sum(ancestry == 1) / len(ancestry)
        prop_pop2 = np.sum(ancestry == 2) / len(ancestry)
        prop_ancestral = np.sum(ancestry == 0) / len(ancestry)
        
        # Count recombination breakpoints (switches between ancestries)
        breakpoints = np.sum(np.diff(ancestry) != 0)
        
        print(f"Sample {i+1}:")
        print(f"  Pop1 ancestry: {prop_pop1*100:.1f}%")
        print(f"  Pop2 ancestry: {prop_pop2*100:.1f}%")
        print(f"  Ancestral: {prop_ancestral*100:.1f}%")
        print(f"  Recombination breakpoints: {breakpoints}")
    
    # Store for visualization
    local_ancestry_data = {
        'windows': windows,
        'ancestry_results': ancestry_results,
        'sample_nodes': selected_samples
    }
    
    print("\n✓ Local ancestry analysis complete")
    
else:
    print("No admixed population samples available for ancestry analysis")

In [ ]:
# Create comprehensive visualization of local ancestry patterns
if 'local_ancestry_data' in locals():
    windows = local_ancestry_data['windows']
    ancestry_results = local_ancestry_data['ancestry_results']
    
    # Create figure with subplots for different interpretations
    # We'll group samples to represent different hybrid scenarios
    fig = plt.figure(figsize=(16, 12))
    
    # Define grid
    gs = fig.add_gridspec(4, 1, hspace=0.4)
    
    n_samples = len(ancestry_results)
    
    # Panel 1: F1-like hybrids (samples with roughly 50-50 ancestry)
    # Select samples with most balanced ancestry
    ancestry_props = []
    for anc in ancestry_results:
        prop1 = np.sum(anc == 1) / len(anc)
        prop2 = np.sum(anc == 2) / len(anc)
        balance = abs(prop1 - prop2)  # Lower is more balanced
        ancestry_props.append((balance, prop1, prop2, anc))
    
    ancestry_props.sort(key=lambda x: x[0])  # Sort by balance
    
    # Panel 1: F1 hybrids (most balanced, if available)
    ax1 = fig.add_subplot(gs[0])
    f1_samples = [x[3] for x in ancestry_props[:min(2, n_samples)]]
    if f1_samples:
        plot_local_ancestry(windows, f1_samples, ax1, 
                          'F1-like Hybrids (Balanced Ancestry ~50:50)',
                          [f'F1-{i+1}' for i in range(len(f1_samples))])
    
    # Panel 2: F2-like (intermediate recombination)
    ax2 = fig.add_subplot(gs[1])
    # Select samples with intermediate number of breakpoints
    breakpoint_counts = []
    for anc in ancestry_results:
        bp_count = np.sum(np.diff(anc) != 0)
        breakpoint_counts.append((bp_count, anc))
    
    breakpoint_counts.sort(key=lambda x: x[0])
    mid_idx = len(breakpoint_counts) // 2
    f2_samples = [x[1] for x in breakpoint_counts[mid_idx:mid_idx+min(2, n_samples)]]
    
    if f2_samples:
        plot_local_ancestry(windows, f2_samples, ax2,
                          'F2-like Hybrids (Increased Recombination)',
                          [f'F2-{i+1}' for i in range(len(f2_samples))])
    
    # Panel 3: Backcross-like (skewed ancestry)
    ax3 = fig.add_subplot(gs[2])
    # Select samples with most skewed ancestry
    ancestry_props_desc = sorted(ancestry_props, key=lambda x: x[0], reverse=True)
    backcross_samples = [x[3] for x in ancestry_props_desc[:min(2, n_samples)]]
    
    if backcross_samples:
        plot_local_ancestry(windows, backcross_samples, ax3,
                          'Backcross-like Hybrids (Skewed Ancestry)',
                          [f'BC-{i+1}' for i in range(len(backcross_samples))])
    
    # Panel 4: Late-generation (most recombination breakpoints)
    ax4 = fig.add_subplot(gs[3])
    late_gen_samples = [x[1] for x in breakpoint_counts[-min(2, n_samples):]]
    
    if late_gen_samples:
        plot_local_ancestry(windows, late_gen_samples, ax4,
                          'Late-Generation Hybrids (Fine-Scale Mosaic)',
                          [f'LG-{i+1}' for i in range(len(late_gen_samples))])
    
    plt.suptitle('Local Ancestry Patterns Across Hybrid Generations', 
                 fontsize=14, fontweight='bold', y=0.995)
    
    plt.tight_layout()
    plt.show()
    
    # Print summary
    print("\nVisualization Summary:")
    print("=" * 60)
    print("The plot shows local ancestry along the genome for different")
    print("types of hybrid individuals:")
    print("")
    print("- F1 hybrids: Large blocks, balanced ancestry")
    print("- F2 hybrids: More recombination breakpoints")
    print("- Backcrosses: Asymmetric ancestry proportions")
    print("- Late-generation: Fine-scale mosaic pattern")
    print("")
    print("Each horizontal bar represents one individual's genome,")
    print("with colors showing which population each segment came from.")
    
else:
    print("No local ancestry data available. Run the previous cell first.")

## Detailed Ancestry Block Analysis

Examine individual samples in more detail to understand ancestry block sizes and recombination patterns.

In [ ]:
# Detailed analysis of ancestry blocks
if 'local_ancestry_data' in locals():
    windows = local_ancestry_data['windows']
    ancestry_results = local_ancestry_data['ancestry_results']
    
    # Analyze block sizes for each sample
    print("Ancestry Block Size Analysis")
    print("=" * 60)
    
    window_size = windows[1] - windows[0]
    
    for i, ancestry in enumerate(ancestry_results):
        # Find runs of the same ancestry
        blocks = []
        current_anc = ancestry[0]
        block_start = 0
        
        for j in range(1, len(ancestry)):
            if ancestry[j] != current_anc:
                # Block ended
                block_length = (j - block_start) * window_size
                blocks.append((current_anc, block_length))
                current_anc = ancestry[j]
                block_start = j
        
        # Add final block
        block_length = (len(ancestry) - block_start) * window_size
        blocks.append((current_anc, block_length))
        
        # Calculate statistics
        pop1_blocks = [b[1] for b in blocks if b[0] == 1]
        pop2_blocks = [b[1] for b in blocks if b[0] == 2]
        
        print(f"\nSample {i+1}:")
        print(f"  Total ancestry blocks: {len(blocks)}")
        
        if pop1_blocks:
            print(f"  Pop1 blocks: {len(pop1_blocks)}")
            print(f"    Mean size: {np.mean(pop1_blocks):,.0f} bp")
            print(f"    Median size: {np.median(pop1_blocks):,.0f} bp")
            print(f"    Max size: {np.max(pop1_blocks):,.0f} bp")
        
        if pop2_blocks:
            print(f"  Pop2 blocks: {len(pop2_blocks)}")
            print(f"    Mean size: {np.mean(pop2_blocks):,.0f} bp")
            print(f"    Median size: {np.median(pop2_blocks):,.0f} bp")
            print(f"    Max size: {np.max(pop2_blocks):,.0f} bp")
    
    # Create histogram of block sizes
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Collect all block sizes
    all_pop1_blocks = []
    all_pop2_blocks = []
    
    for ancestry in ancestry_results:
        blocks = []
        current_anc = ancestry[0]
        block_start = 0
        
        for j in range(1, len(ancestry)):
            if ancestry[j] != current_anc:
                block_length = (j - block_start) * window_size
                if current_anc == 1:
                    all_pop1_blocks.append(block_length)
                elif current_anc == 2:
                    all_pop2_blocks.append(block_length)
                current_anc = ancestry[j]
                block_start = j
        
        block_length = (len(ancestry) - block_start) * window_size
        if current_anc == 1:
            all_pop1_blocks.append(block_length)
        elif current_anc == 2:
            all_pop2_blocks.append(block_length)
    
    # Plot histograms
    if all_pop1_blocks:
        ax1.hist(np.array(all_pop1_blocks)/1000, bins=30, color='#3498db', 
                alpha=0.7, edgecolor='black')
        ax1.set_xlabel('Block Size (kb)', fontsize=11, fontweight='bold')
        ax1.set_ylabel('Frequency', fontsize=11, fontweight='bold')
        ax1.set_title('Population 1 Ancestry Block Sizes', fontsize=12, fontweight='bold')
        ax1.grid(alpha=0.3)
    
    if all_pop2_blocks:
        ax2.hist(np.array(all_pop2_blocks)/1000, bins=30, color='#e74c3c', 
                alpha=0.7, edgecolor='black')
        ax2.set_xlabel('Block Size (kb)', fontsize=11, fontweight='bold')
        ax2.set_ylabel('Frequency', fontsize=11, fontweight='bold')
        ax2.set_title('Population 2 Ancestry Block Sizes', fontsize=12, fontweight='bold')
        ax2.grid(alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print("\n✓ Block size analysis complete")
    
else:
    print("No local ancestry data available. Run the previous cells first.")

## Tract Length Decay Over Time

Analyze how admixture tract lengths decrease over time due to recombination.

### Theory

After admixture, recombination progressively breaks up ancestry blocks. The mean length of ancestry tracts decreases exponentially with time according to:

**L = 1 / [(1-h) × r × (t-1)]**

Where:
- **L** = mean tract length (in base pairs)
- **h** = hybrid index (proportion of genome from population 1)
- **r** = recombination rate per base pair per generation
- **t** = number of generations since admixture formation

### Visualization

This analysis:
1. Calculates observed mean tract lengths for population 1 ancestry
2. Compares observed data with analytical predictions
3. Shows the rapid decay of tract length through time
4. Demonstrates how recombination rate affects tract length dynamics

In [ ]:
# Calculate observed tract lengths for population 1 ancestry
if 'local_ancestry_data' in locals():
    print("Analyzing tract length decay over time...")
    print("=" * 60)
    
    windows = local_ancestry_data['windows']
    ancestry_results = local_ancestry_data['ancestry_results']
    window_size = windows[1] - windows[0]
    
    # Collect all tract lengths for population 1
    pop1_tract_lengths = []
    
    for ancestry in ancestry_results:
        # Find runs of population 1 ancestry
        current_anc = ancestry[0]
        block_start = 0
        
        for j in range(1, len(ancestry)):
            if ancestry[j] != current_anc:
                # Block ended
                if current_anc == 1:  # Population 1 ancestry
                    block_length = (j - block_start) * window_size
                    pop1_tract_lengths.append(block_length)
                current_anc = ancestry[j]
                block_start = j
        
        # Add final block
        if current_anc == 1:
            block_length = (len(ancestry) - block_start) * window_size
            pop1_tract_lengths.append(block_length)
    
    # Calculate statistics
    if pop1_tract_lengths:
        observed_mean_length = np.mean(pop1_tract_lengths)
        observed_median_length = np.median(pop1_tract_lengths)
        
        print(f"Population 1 Ancestry Tracts:")
        print(f"  Number of tracts: {len(pop1_tract_lengths)}")
        print(f"  Mean length: {observed_mean_length:,.0f} bp")
        print(f"  Median length: {observed_median_length:,.0f} bp")
        print(f"  Min length: {np.min(pop1_tract_lengths):,.0f} bp")
        print(f"  Max length: {np.max(pop1_tract_lengths):,.0f} bp")
        
        # Calculate hybrid index (h) - proportion of genome from pop1
        total_pop1_ancestry = 0
        total_windows = 0
        
        for ancestry in ancestry_results:
            total_pop1_ancestry += np.sum(ancestry == 1)
            total_windows += len(ancestry)
        
        h = total_pop1_ancestry / total_windows
        print(f"\nHybrid index (h): {h:.3f}")
        print(f"  (Proportion of genome from Population 1)")
        
        # Calculate time since admixture
        # In our simulation, admixture occurs at Tadmix, and we sample at Tend
        t = Tend - Tadmix
        print(f"\nTime since admixture (t): {t} generations")
        
        # Calculate analytical prediction
        # L = 1 / [(1-h) * r * (t-1)]
        if t > 1 and h < 1.0:
            predicted_mean_length = 1.0 / ((1 - h) * recomb_rate * (t - 1))
            
            print(f"\nAnalytical Prediction:")
            print(f"  Formula: L = 1 / [(1-h) × r × (t-1)]")
            print(f"  Predicted mean length: {predicted_mean_length:,.0f} bp")
            print(f"  Observed/Predicted ratio: {observed_mean_length/predicted_mean_length:.3f}")
            
            # Store results for visualization
            tract_length_data = {
                'pop1_tract_lengths': pop1_tract_lengths,
                'observed_mean': observed_mean_length,
                'observed_median': observed_median_length,
                'h': h,
                't': t,
                'predicted_mean': predicted_mean_length,
                'recomb_rate': recomb_rate
            }
        else:
            print("\n⚠ Cannot calculate prediction: t must be > 1 and h < 1.0")
            tract_length_data = {
                'pop1_tract_lengths': pop1_tract_lengths,
                'observed_mean': observed_mean_length,
                'observed_median': observed_median_length,
                'h': h,
                't': t
            }
        
        print("\n✓ Tract length analysis complete")
    else:
        print("⚠ No Population 1 ancestry tracts found")
else:
    print("No local ancestry data available. Run previous cells first.")

In [ ]:
# Visualize tract length decay over time
if 'tract_length_data' in locals():
    print("Creating tract length decay visualization...")
    
    # Create figure with two subplots
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    
    # Panel 1: Distribution of observed tract lengths
    pop1_tracts = tract_length_data['pop1_tract_lengths']
    observed_mean = tract_length_data['observed_mean']
    
    # Convert to kb for better readability
    pop1_tracts_kb = np.array(pop1_tracts) / 1000
    
    ax1.hist(pop1_tracts_kb, bins=50, color='#3498db', alpha=0.7, 
             edgecolor='black', linewidth=0.5)
    ax1.axvline(observed_mean/1000, color='red', linestyle='--', linewidth=2,
                label=f'Observed Mean = {observed_mean/1000:.1f} kb')
    
    if 'predicted_mean' in tract_length_data:
        predicted_mean = tract_length_data['predicted_mean']
        ax1.axvline(predicted_mean/1000, color='green', linestyle='--', linewidth=2,
                    label=f'Predicted Mean = {predicted_mean/1000:.1f} kb')
    
    ax1.set_xlabel('Tract Length (kb)', fontsize=12, fontweight='bold')
    ax1.set_ylabel('Frequency', fontsize=12, fontweight='bold')
    ax1.set_title('Distribution of Population 1 Ancestry Tract Lengths',
                  fontsize=13, fontweight='bold')
    ax1.legend(loc='upper right', fontsize=10)
    ax1.grid(alpha=0.3, linestyle='--')
    
    # Panel 2: Tract length vs time with analytical prediction
    if 'predicted_mean' in tract_length_data:
        h = tract_length_data['h']
        t_current = tract_length_data['t']
        r = tract_length_data['recomb_rate']
        
        # Generate predictions for a range of times
        t_values = np.arange(2, max(t_current + 10, 20))
        predicted_lengths = 1.0 / ((1 - h) * r * (t_values - 1))
        
        # Plot analytical prediction curve
        ax2.plot(t_values, predicted_lengths / 1000, 'g-', linewidth=2.5, 
                label='Analytical Prediction\nL = 1/[(1-h)r(t-1)]')
        
        # Plot observed data point
        ax2.plot(t_current, observed_mean / 1000, 'ro', markersize=12, 
                label=f'Observed (t={t_current})', markeredgecolor='black', 
                markeredgewidth=1.5, zorder=5)
        
        # Add annotation
        ax2.annotate(f'{observed_mean/1000:.1f} kb', 
                    xy=(t_current, observed_mean/1000),
                    xytext=(t_current + 1, observed_mean/1000 + 20),
                    fontsize=10, fontweight='bold',
                    arrowprops=dict(arrowstyle='->', color='black', lw=1.5))
        
        ax2.set_xlabel('Generations Since Admixture (t)', fontsize=12, fontweight='bold')
        ax2.set_ylabel('Mean Tract Length (kb)', fontsize=12, fontweight='bold')
        ax2.set_title('Tract Length Decay Over Time', fontsize=13, fontweight='bold')
        ax2.legend(loc='upper right', fontsize=10, framealpha=0.9)
        ax2.grid(alpha=0.3, linestyle='--')
        
        # Add text box with parameters
        textstr = f'h = {h:.3f}\nr = {r:.2e}\nt = {t_current}'
        props = dict(boxstyle='round', facecolor='wheat', alpha=0.8)
        ax2.text(0.05, 0.95, textstr, transform=ax2.transAxes, fontsize=10,
                verticalalignment='top', bbox=props)
        
        # Set y-axis to start at 0 for better visualization
        ax2.set_ylim(bottom=0)
        
    else:
        ax2.text(0.5, 0.5, 'Analytical prediction not available\n(t must be > 1 and h < 1.0)',
                ha='center', va='center', transform=ax2.transAxes, fontsize=12)
        ax2.set_xlabel('Generations Since Admixture (t)', fontsize=12, fontweight='bold')
        ax2.set_ylabel('Mean Tract Length (kb)', fontsize=12, fontweight='bold')
        ax2.set_title('Tract Length Decay Over Time', fontsize=13, fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    # Print interpretation
    print("\nInterpretation:")
    print("=" * 60)
    print("The left panel shows the distribution of observed tract lengths.")
    print("The right panel shows how mean tract length decreases with time.")
    print("")
    print("Key observations:")
    print(f"  • Mean tract length decays as 1/(t-1)")
    print(f"  • Recombination rate (r={r:.2e}) determines decay speed")
    print(f"  • Hybrid index (h={h:.3f}) affects the effective rate")
    print("  • Rapid decay in early generations, then slows")
    
    if 'predicted_mean' in tract_length_data:
        ratio = observed_mean / tract_length_data['predicted_mean']
        if 0.8 <= ratio <= 1.2:
            print(f"  • Good agreement with theory (ratio = {ratio:.2f})")
        else:
            print(f"  • Some deviation from theory (ratio = {ratio:.2f})")
            print("    This can occur due to stochastic effects or window resolution")
    
    print("\n✓ Visualization complete")
    
else:
    print("No tract length data available. Run the previous cell first.")

## Tract Length Through Multiple Generations (Optional)

To better visualize the decay pattern, we can simulate tract lengths at multiple time points.

This cell demonstrates how to run the simulation with different end times to capture the full decay curve. This is optional and can be time-consuming.

In [ ]:
# Multi-generation tract length analysis (optional, time-consuming)
run_multi_generation = False  # Set to True to run this analysis

if run_multi_generation:
    print("Running multi-generation tract length analysis...")
    print("This may take several minutes...\n")
    
    import tempfile
    
    # Define time points to analyze
    time_points = [1, 2, 3, 5, 10, 20, 50]  # Generations after admixture
    time_points = [t for t in time_points if t <= Tend - Tadmix]  # Only valid times
    
    observed_means = []
    predicted_means = []
    
    for delta_t in time_points:
        print(f"Processing t = {delta_t} generations...", end='\r')
        
        # Create temporary output file
        temp_end = Tadmix + delta_t
        temp_output = f"temp_time_{delta_t}.trees"
        
        # Generate SLiM script for this time point
        slim_script_temp = f"""
initialize() {{
    initializeTreeSeq();
    initializeMutationRate(0.0);
    initializeMutationType("m1", 0.5, "f", 0.0);
    initializeGenomicElementType("g1", m1, 1.0);
    initializeGenomicElement(g1, 0, {int(genome_length)-1});
    initializeRecombinationRate({recomb_rate});
}}

1 {{
    sim.addSubpop("p0", {Na});
}}

{Tsplit} {{
    sim.addSubpopSplit("p1", {N1}, p0);
    sim.addSubpopSplit("p2", {N2}, p0);
    p0.setSubpopulationSize(0);
}}

{Tadmix} {{
    sim.addSubpop("p3", {NAdmix});
    p3.setMigrationRates(c(p1, p2), c({f}, {1-f}));
}}

{Tadmix + 1} {{
    p3.setMigrationRates(c(p1, p2), c(0.0, 0.0));
}}

{temp_end} late() {{
    sim.treeSeqOutput("{temp_output}");
    sim.simulationFinished();
}}
"""
        
        # Save and run
        temp_slim = f"temp_time_{delta_t}.slim"
        with open(temp_slim, 'w') as f:
            f.write(slim_script_temp)
        
        try:
            result = subprocess.run(
                ['slim', temp_slim],
                capture_output=True,
                text=True,
                timeout=120
            )
            
            if result.returncode == 0 and os.path.exists(temp_output):
                # Load and analyze
                ts_temp = tskit.load(temp_output)
                
                # Get admixed samples
                temp_samples = []
                for node in ts_temp.nodes():
                    if node.population == 3 and node.is_sample():
                        temp_samples.append(node.id)
                
                if len(temp_samples) > 0:
                    # Analyze a few samples
                    sample_tracts = []
                    for sample in temp_samples[:5]:  # Use 5 samples
                        windows, ancestry = get_local_ancestry(ts_temp, sample, 
                                                               pop1_id=1, pop2_id=2, 
                                                               num_windows=200)
                        
                        # Extract pop1 tract lengths
                        window_size = windows[1] - windows[0]
                        current_anc = ancestry[0]
                        block_start = 0
                        
                        for j in range(1, len(ancestry)):
                            if ancestry[j] != current_anc:
                                if current_anc == 1:
                                    block_length = (j - block_start) * window_size
                                    sample_tracts.append(block_length)
                                current_anc = ancestry[j]
                                block_start = j
                        
                        if current_anc == 1:
                            block_length = (len(ancestry) - block_start) * window_size
                            sample_tracts.append(block_length)
                    
                    if sample_tracts:
                        obs_mean = np.mean(sample_tracts)
                        observed_means.append(obs_mean)
                        
                        # Calculate prediction
                        if delta_t > 1:
                            pred_mean = 1.0 / ((1 - f) * recomb_rate * (delta_t - 1))
                            predicted_means.append(pred_mean)
                        else:
                            predicted_means.append(np.nan)
                    else:
                        observed_means.append(np.nan)
                        predicted_means.append(np.nan)
                
                # Clean up
                os.remove(temp_output)
            
            os.remove(temp_slim)
            
        except Exception as e:
            print(f"\nError at t={delta_t}: {e}")
            observed_means.append(np.nan)
            predicted_means.append(np.nan)
    
    print(f"✓ Completed analysis for {len(time_points)} time points" + " " * 30)
    
    # Create comprehensive plot
    if observed_means:
        fig, ax = plt.subplots(figsize=(12, 7))
        
        # Generate smooth prediction curve
        t_smooth = np.linspace(2, max(time_points), 100)
        pred_smooth = 1.0 / ((1 - f) * recomb_rate * (t_smooth - 1))
        
        # Plot prediction
        ax.plot(t_smooth, pred_smooth / 1000, 'g-', linewidth=3, 
               label='Analytical Prediction: L = 1/[(1-h)r(t-1)]', alpha=0.8)
        
        # Plot observed data
        valid_idx = ~np.isnan(observed_means)
        ax.plot(np.array(time_points)[valid_idx], 
               np.array(observed_means)[valid_idx] / 1000, 
               'ro-', markersize=10, linewidth=2, 
               label='Observed Mean Tract Length',
               markeredgecolor='black', markeredgewidth=1.5)
        
        ax.set_xlabel('Generations Since Admixture (t)', fontsize=13, fontweight='bold')
        ax.set_ylabel('Mean Tract Length (kb)', fontsize=13, fontweight='bold')
        ax.set_title('Rapid Decay of Admixture Tract Length Through Time',
                    fontsize=14, fontweight='bold')
        ax.legend(loc='upper right', fontsize=11, framealpha=0.9)
        ax.grid(alpha=0.3, linestyle='--')
        ax.set_ylim(bottom=0)
        
        plt.tight_layout()
        plt.show()
        
        print("\n✓ Multi-generation analysis complete")
        
else:
    print("Multi-generation analysis disabled.")
    print("Set run_multi_generation = True to enable this analysis.")
    print("Note: This will run multiple SLiM simulations and may take time.")

## Summary Statistics Table

Compile all key statistics into a summary table.

In [ ]:
# Create comprehensive summary
print("Simulation Summary")
print("=" * 80)

summary_data = []

for pop_id in range(ts.num_populations):
    samples = pop_samples.get(pop_id, [])
    if len(samples) >= 2:
        pi = ts.diversity(sample_sets=[samples], mode='site')
        pi_value = pi[0] if len(pi) > 0 else np.nan
        
        pop_name = f"Pop{pop_id}"
        if pop_id == 0:
            pop_name = "Ancestral (p0)"
        elif pop_id == 1:
            pop_name = "Population 1 (p1)"
        elif pop_id == 2:
            pop_name = "Population 2 (p2)"
        elif pop_id == 3:
            pop_name = "Admixed (p3)"
        
        summary_data.append({
            'Population': pop_name,
            'ID': pop_id,
            'Samples': len(samples),
            'Diversity (π)': f"{pi_value:.6f}" if not np.isnan(pi_value) else "N/A"
        })

df_summary = pd.DataFrame(summary_data)
print(df_summary.to_string(index=False))

print("\n" + "=" * 80)
print("Simulation completed successfully!")
print(f"Tree sequence saved to: {output_trees}")
print(f"SLiM script saved to: {slim_script_file}")

## Cleanup (Optional)

Remove temporary files if desired.

In [ ]:
# Uncomment to remove output files
# import os
# if os.path.exists(output_trees):
#     os.remove(output_trees)
#     print(f"Removed {output_trees}")
# if os.path.exists(slim_script_file):
#     os.remove(slim_script_file)
#     print(f"Removed {slim_script_file}")

print("Files retained for further analysis.")